# 03 · Modelling results: tables and figures from a fitted run

Regenerates every **model-derived** table and figure from the files that
[`02_modelling.ipynb`](02_modelling.ipynb) wrote to `results/{run_tag}/` — no model object
is needed. For each metric the notebook loads the **best-performing model of the run**
(`MODEL_DIRS[metric] = None`, resolved with `pf.resolve_model_dir`) or a **specific model
folder** you point it at. Model-free outputs (demographics tables, demographic
distributions, observed maps, thinning validation) come from [`01_cohort_stats.ipynb`](01_cohort_stats.ipynb).

**The executed notebook is the report**: every table and figure is rendered inline and
nothing is written to disk (apart from the optional full-cohort random intercepts, which are
stored next to the model). Set `SAVE_PAPER_ASSETS = True` in `CONFIG` to additionally write
the publication assets (`.tex` tables, 300-dpi PNG + SVG + PDF figures) to `results/paper/`.

| Output | Function | Asset (when saved) |
| :--- | :--- | :--- |
| Results tables (main text + supplementary: stage winners, full grid, OLS parameters) | `paper_tables.generate_all_tables` | `results_{metric}.tex`, `supp_*.tex`, `{metric}_full_grid.tex`, `{metric}_ols_params.tex` |
| Full-cohort random intercepts of the chosen model (needed by the "accuracy" figures) | `pf.extend_random_effects_to_cohort` | `<model folder>/random_effects.parquet` |
| fig:{metric}_residuals, fig:{metric}_qq | `pf.make_model_residuals_plot`, `pf.make_random_intercepts_plot` | `{metric}_residuals`, `{metric}_qq` |
| fig:lat_bin_fits, fig:countries_prediction_fit | `pf.make_combined_seasonal_fit_lat_bin / _country` | `rhr_hrv_combined_seasonal_prediction_fit_*` |
| fig:demographic_fit | `pf.make_joint_demographics_calibration` | `demographic_fit` |
| fig:global_residuals_grid | `pf.make_world_contour_residuals` | `world_contour_combined_residuals` |

The tables describe the whole competition of the run (every candidate plus the REML
winner); the figures use the model folder selected in `CONFIG`. The combined RHR + HRV
figures run when both metrics are configured. Section 2 compares two runs model by model
and Section 3 browses the per-model diagnostics.

The world residual contours need the Natural Earth file described in `data/README.md` in
`data/extra_data/`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import os
import sys
import time

from IPython.display import HTML, Markdown, SVG, display
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # library modules (public release)
import file_io as fio
import paper_tables
import publication_figures as pf
import seasonality_lib as sl

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
METRICS = ('rhr', 'hrv')  # metrics to process (combined figures need both)
# results/ sub-folder per metric. None = results/{metric} when present, otherwise
# the default run tag of 02_modelling.ipynb (rhr_rerun, hrv_rerun).
RUN_TAGS = {'rhr': None, 'hrv': None}
# Per metric: None = best-performing model of the run (BIC winner, REML refit);
# or a model folder name inside results/{run_tag}/ (or an absolute path).
MODEL_DIRS = {'rhr': None, 'hrv': None}
# Also write the publication assets (.tex, png/svg/pdf) to results/paper/?
# Otherwise this notebook is the report.
SAVE_PAPER_ASSETS = False
EXTEND_BLUPS = (
    True  # full-cohort random intercepts for the chosen model (see below)
)
RANDOM_STATE = 42

# Runs whose winner folders already hold full-cohort random intercepts and are
# never written into (none by default).
PROTECTED_RUN_TAGS = ()

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
results_root = os.path.join(data_dir, fio.RESULTS_SUBDIR)
out_dir = fio.get_paper_dir(data_dir) if SAVE_PAPER_ASSETS else None
if out_dir:
  fio.make_dirs(out_dir)
pf.setup_nature_style()


def show_table(label, latex):
  """Renders a generated LaTeX table inline (preview + collapsible source)."""
  preview = paper_tables.table_preview(latex)
  display(Markdown(f'**`{label}`** — {preview.attrs.get("caption", "")}'))
  display(HTML(preview.to_html(index=False)))
  if preview.attrs.get('notes'):
    display(Markdown(f'<small>{preview.attrs["notes"]}</small>'))
  display(
      HTML(
          '<details><summary>LaTeX source</summary><pre style="font-size:80%">'
          + latex.replace('&', '&amp;').replace('<', '&lt;')
          + '</pre></details>'
      )
  )


def resolve_run_tag(metric, tag):
  if tag is not None:
    return tag
  for candidate in (metric, f'{metric}_rerun'):
    if fio.file_exists(fio.get_results_dir(candidate, data_dir)):
      return candidate
  raise FileNotFoundError(
      f'no results/{metric} or results/{metric}_rerun under {data_dir}: '
      'run 02_modelling.ipynb first or set RUN_TAGS'
  )


RUN_TAGS = {m: resolve_run_tag(m, RUN_TAGS.get(m)) for m in METRICS}
run_dirs = {m: fio.get_results_dir(RUN_TAGS[m], data_dir) for m in METRICS}
model_dirs = {
    m: pf.resolve_model_dir(data_dir, m, RUN_TAGS[m], MODEL_DIRS[m])
    for m in METRICS
}
for m in METRICS:
  print(
      f'{m.upper()}: run {RUN_TAGS[m]!r}\n  model folder:'
      f' {os.path.basename(model_dirs[m])}'
  )
print('publication assets ->', out_dir or 'not saved (SAVE_PAPER_ASSETS=False)')

## 1. Per-metric outputs

In [ ]:
for metric in METRICS:
  run_tag, run_dir, model_dir = (
      RUN_TAGS[metric],
      run_dirs[metric],
      model_dirs[metric],
  )
  print(f'\n{"=" * 70}\n{metric.upper()}  ({run_tag})\n{"=" * 70}')

  # Stage winners of the run (lowest BIC per stage) -> main-text + supplementary tables.
  results_df = paper_tables.load_model_results(data_dir, metric, run_tag)
  try:
    stage_winners = paper_tables.infer_stage_winners(results_df)
  except KeyError as e:  # partial run (MAX_STAGE < 6) lacks some stages
    print(f'Tables skipped: {e}')
  else:
    print('Stage winners:', stage_winners)
    tables = paper_tables.generate_all_tables(
        data_dir,
        out_dir=out_dir,
        metrics=(metric,),
        results_subdirs={metric: run_tag},
        winners={metric: stage_winners},
    )
    for label, latex in tables.items():
      show_table(label, latex)

  # Random intercepts for EVERY user of the clean cohort (empirical-Bayes BLUPs from the
  # model's fixed effects + variance components). 02_modelling only stores intercepts for
  # the fitting subsample; the accuracy-mode figures below plot random users. Skipped for
  # PROTECTED_RUN_TAGS and for folders extended earlier.
  if EXTEND_BLUPS and run_tag not in PROTECTED_RUN_TAGS:
    t0 = time.time()
    pf.extend_random_effects_to_cohort(
        data_dir, run_dir, metric, model_dir=model_dir
    )
    print(f'  ({time.time() - t0:.0f}s)')
    gc.collect()

  # fig:{metric}_residuals and fig:{metric}_qq of the chosen model.
  pf.make_model_residuals_plot(
      data_dir,
      results_root,
      metric,
      out_dir,
      subdir=run_tag,
      model_dir=model_dir,
  )
  pf.make_random_intercepts_plot(
      results_root, metric, out_dir, subdir=run_tag, model_dir=model_dir
  )
  plt.show()
  plt.close('all')

### Combined RHR + HRV figures

Each maker reloads the cohort data of both metrics (memory-bound, one metric at a time),
rebuilds the model features and predicts with the chosen model's coefficients and random
intercepts.

In [ ]:
if set(METRICS) >= {'rhr', 'hrv'}:
  kw = dict(rhr_subdir=RUN_TAGS['rhr'], hrv_subdir=RUN_TAGS['hrv'])
  pf.make_combined_seasonal_fit_lat_bin(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # fig:lat_bin_fits
  pf.make_combined_seasonal_fit_country(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # fig:countries_prediction_fit
  pf.make_joint_demographics_calibration(
      data_dir, results_root, out_dir, model_dirs=model_dirs, **kw
  )  # fig:demographic_fit
  # fig:global_residuals_grid (lowest-BIC row) needs the Natural Earth geojson in extra_data/
  # (see data/README.md); skipped when it is not available.
  if fio.file_exists(os.path.join(extra_dir, pf.WORLD_GEOJSON_FILE)):
    pf.make_world_contour_residuals(
        data_dir, results_root, extra_dir, out_dir, **kw
    )
  else:
    print(
        f'{pf.WORLD_GEOJSON_FILE} not found in {extra_dir}: world contour'
        ' figure skipped'
    )
  plt.show()
  plt.close('all')
  if out_dir:
    print('Combined figures written to', out_dir)
else:
  print('Combined figures need both metrics in METRICS.')

## 2. Compare two runs (optional)

Use this to check reproducibility, e.g. a rerun against an earlier run of the same metric.
Models are matched on their **formula** (whitespace-insensitive), so renamed stages still
line up. Differences are `run - baseline`; the winner coefficients are compared in units of
the baseline standard error.

In [ ]:
COMPARE_METRIC = 'rhr'
COMPARE_RUN_TAG = RUN_TAGS[COMPARE_METRIC]  # e.g. 'rhr_rerun'
COMPARE_BASELINE_TAG = COMPARE_METRIC  # baseline run of the same metric

FIT_COLS = [
    'BIC',
    'AIC',
    'log_likelihood',
    'marginal_R2',
    'conditional_R2',
    'ICC',
    'rmse',
    'aggregate_R_country',
    'aggregate_R_lat_bin',
    'aggregate_R_age_group',
]
# Coefficient tables may carry statsmodels summary headers; normalise both
# layouts to the snake_case names used below.
COEF_COLS = {
    'Coef.': 'coef',
    'Std.Err.': 'std_err',
    'P>|z|': 'P_z',
    '[0.025': 'ci_lower',
    '0.975]': 'ci_upper',
}


def _read_csv(path, **kwargs):
  with fio.open_file(path, 'r') as f:
    return pd.read_csv(f, **kwargs)


def list_model_dirs(run_dir):
  """Per-model folders of a run (those containing model-row.csv), sorted by name."""
  return sorted(
      d
      for d in fio.list_dir(run_dir)
      if fio.file_exists(os.path.join(run_dir, d, 'model-row.csv'))
  )


def load_run_table(run_dir, metric):
  path = sl.results_csv_path(run_dir, metric)
  if fio.file_exists(path):
    df = _read_csv(path)
  else:
    df = pd.concat(
        [
            _read_csv(os.path.join(run_dir, d, 'model-row.csv')).assign(
                model_dir=d
            )
            for d in list_model_dirs(run_dir)
        ],
        ignore_index=True,
    )
  df['formula_key'] = df['formula'].astype(str).str.replace(' ', '')
  return df.drop_duplicates('formula_key')


def compare_runs(run_dir, baseline_dir, metric, cols=FIT_COLS):
  a, b = load_run_table(run_dir, metric), load_run_table(baseline_dir, metric)
  cols = [c for c in cols if c in a.columns and c in b.columns]
  merged = a[['model', 'formula_key'] + cols].merge(
      b[['model', 'formula_key'] + cols],
      on='formula_key',
      suffixes=('', '_base'),
      how='outer',
  )
  for c in cols:
    merged[f'd_{c}'] = merged[c] - merged[f'{c}_base']
  order = ['model', 'model_base'] + [
      x for c in cols for x in (c, f'{c}_base', f'd_{c}')
  ]
  return merged[order].sort_values('BIC')


def load_coefs(run_dir, metric):
  path = sl.coefficients_csv_path(run_dir, metric)
  return (
      _read_csv(path, index_col=0).rename(columns=COEF_COLS)
      if fio.file_exists(path)
      else None
  )


cmp_run_dir = fio.get_results_dir(COMPARE_RUN_TAG, data_dir)
cmp_base_dir = fio.get_results_dir(COMPARE_BASELINE_TAG, data_dir)
if COMPARE_RUN_TAG == COMPARE_BASELINE_TAG:
  print(
      'Set COMPARE_RUN_TAG and COMPARE_BASELINE_TAG to two different runs to'
      ' compare them.'
  )
elif fio.file_exists(cmp_run_dir) and fio.file_exists(cmp_base_dir):
  comparison = compare_runs(cmp_run_dir, cmp_base_dir, COMPARE_METRIC)
  display(comparison)
  print(
      'Lowest-BIC model  run / baseline:',
      comparison.iloc[0]['model'],
      '/',
      comparison.sort_values('BIC_base').iloc[0]['model_base'],
  )
  ca, cb = load_coefs(cmp_run_dir, COMPARE_METRIC), load_coefs(
      cmp_base_dir, COMPARE_METRIC
  )
  if ca is not None and cb is not None:
    coef_cmp = ca[['coef', 'std_err']].join(
        cb[['coef', 'std_err']], how='outer', rsuffix='_base'
    )
    coef_cmp['d_coef'] = coef_cmp['coef'] - coef_cmp['coef_base']
    coef_cmp['d_in_se'] = coef_cmp['d_coef'] / coef_cmp['std_err_base']
    display(coef_cmp.round(5))
    print(
        'Max |delta| in baseline SE units:',
        coef_cmp['d_in_se'].abs().max().round(2),
    )

## 3. Browse the diagnostics of the chosen model folders

In [ ]:
def show_model_folder(
    model_dir,
    svg_names=(
        'aggregate_validation_lat_bin',
        'aggregate_validation_country',
        'prediction_fit_lat_bin',
        'residuals',
    ),
):
  """Prints the CSV summaries and renders selected SVGs of one model folder."""
  print(f'\n--- {os.path.basename(model_dir)}')
  for name in ('model-row.csv', 'vif.csv'):
    path = os.path.join(model_dir, name)
    if fio.file_exists(path):
      display(_read_csv(path).T if name == 'model-row.csv' else _read_csv(path))
  for name in svg_names:
    path = os.path.join(model_dir, f'{name}.svg')
    if fio.file_exists(path):
      print(f'--- {name}')
      with fio.open_file(path, 'r') as f:
        display(SVG(data=f.read()))


for metric in METRICS:
  show_model_folder(model_dirs[metric])